### 🧠 What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [1]:
## Import necessary libraries and modules
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model

from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

## Step 1: Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

## Step 2: Create Vector Stores
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type = "mmr", search_kwargs = {"k": 4, "lambda_mult": 0.7})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
## Step 3 : LLM and Prompt
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = init_chat_model(model = "openai/gpt-oss-120b", model_provider = "groq", temperature = 0.4)
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000024F76312BA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000024F7614C050>, model_name='openai/gpt-oss-120b', temperature=0.4, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
## Step 3: Query decomposition
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")

decomposition_chain = decomposition_prompt | llm | StrOutputParser()

query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question = decomposition_chain.invoke({"question": query})

print(decomposition_question)

**Sub‑questions**

1. What memory mechanisms does LangChain offer (e.g., conversation buffers, vector stores, summary memory) and how are they implemented?  
2. How are agents designed and used in LangChain (tool‑calling, routing, planning, etc.)?  
3. What memory capabilities are provided by CrewAI and how does CrewAI manage state across tasks?  
4. In what ways do LangChain’s memory and agent architectures differ from or resemble those of CrewAI?


In [6]:
## Step 4: QA chain per sub-question
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")

## Create the document chain
qa_chain = create_stuff_documents_chain(llm=llm, prompt = qa_prompt)

In [7]:
## Step 5: Full RAG pipeline logic
def full_query_decomposition_rag_pipeline(user_query):
    
    ## Decompose the query
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]
    
    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")
    
    return "\n\n".join(results)

## Step 6: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: **Sub‑questions**
A: Below are several concrete sub‑questions you could ask to explore the information in the provided context more deeply:

1. **Knowledge Injection**
   - How does injecting fetched knowledge into the LLM prompt reduce hallucination?
   - What mechanisms does CrewAI use to fetch and inject external knowledge into prompts?
   - Are there any best‑practice guidelines for deciding which knowledge to inject and when?

2. **Agent Context‑Sharing (CrewAI)**
   - What is the structure of the intermediate data that agents share with one another?
   - How does context‑sharing enable emergent behaviors such as delegation, consultation, and review?
   - Can you give an example workflow that demonstrates an agent delegating a task to another agent via context‑sharing?
   - How does CrewAI handle conflicts or inconsistencies when multiple agents provide differing intermediate data?

3. **Scalability of CrewAI**
   - In what ways does CrewAI support horizontal s